# STH-SOPR Only Entry Test

Testing simplified entry: just STH-SOPR < 1 with 30% trail

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data/raw")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
df = df[df.index >= '2019-01-01'].dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
# Define entry conditions
strat002_cond = (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['rl_zscore'] > 0.5)
strat002_entries = strat002_cond & ~strat002_cond.shift(1).fillna(False)

sth_only_cond = df['sopr_sth'] < 1
sth_only_entries = sth_only_cond & ~sth_only_cond.shift(1).fillna(False)

print("ENTRY SIGNAL COMPARISON")
print("="*60)
print(f"STRAT-002 (triple filter): {strat002_entries.sum()} entries ({strat002_entries.sum()/7:.1f}/year)")
print(f"STH-SOPR < 1 only:         {sth_only_entries.sum()} entries ({sth_only_entries.sum()/7:.1f}/year)")

In [ ]:
def run_backtest(data, entries, trail=0.30, init_cash=100000):
    if entries.sum() == 0:
        return None
    return vbt.Portfolio.from_signals(
        close=data['price'],
        entries=entries,
        sl_stop=trail,
        sl_trail=True,
        stop_exit_price='close',
        fees=0.001,
        init_cash=init_cash,
        freq='D'
    )

In [ ]:
# Run both backtests
pf_strat002 = run_backtest(df, strat002_entries, 0.30)
pf_sth_only = run_backtest(df, sth_only_entries, 0.30)

print("BACKTEST COMPARISON (30% trailing stop)")
print("="*90)
print(f"{'Metric':<25} {'STRAT-002 (triple)':>25} {'STH-SOPR only':>25}")
print("-"*90)

metrics = [
    ('Total Return', lambda pf: f"{pf.total_return()*100:+,.0f}%"),
    ('CAGR', lambda pf: f"{pf.annualized_return()*100:+,.1f}%"),
    ('Sharpe Ratio', lambda pf: f"{pf.sharpe_ratio():.2f}"),
    ('Sortino Ratio', lambda pf: f"{pf.sortino_ratio():.2f}"),
    ('Max Drawdown', lambda pf: f"{pf.max_drawdown()*100:.1f}%"),
    ('Win Rate', lambda pf: f"{pf.trades.win_rate()*100:.0f}%"),
    ('Total Trades', lambda pf: f"{pf.trades.count()}"),
    ('Trades/Year', lambda pf: f"{pf.trades.count()/7:.1f}"),
    ('Avg Win', lambda pf: f"{pf.trades.records[pf.trades.records['return'] > 0]['return'].mean()*100:+.0f}%" if len(pf.trades.records[pf.trades.records['return'] > 0]) > 0 else "N/A"),
    ('Avg Loss', lambda pf: f"{pf.trades.records[pf.trades.records['return'] <= 0]['return'].mean()*100:.0f}%" if len(pf.trades.records[pf.trades.records['return'] <= 0]) > 0 else "N/A"),
    ('Profit Factor', lambda pf: f"{pf.trades.profit_factor():.2f}"),
    ('Final Value', lambda pf: f"${pf.final_value():,.0f}"),
]

for name, func in metrics:
    try:
        v1 = func(pf_strat002)
        v2 = func(pf_sth_only)
        print(f"{name:<25} {v1:>25} {v2:>25}")
    except:
        pass

bh = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100
print("-"*90)
print(f"{'Buy & Hold':<25} {bh:>+24.0f}%")

In [ ]:
# Test different trail stops for STH-SOPR only
print("\nSTH-SOPR ONLY WITH DIFFERENT TRAIL STOPS")
print("="*100)
print(f"{'Trail %':<10} {'Return':>12} {'Sharpe':>10} {'MaxDD':>10} {'Trades':>10} {'/Year':>8} {'Win%':>8}")
print("-"*100)

for trail in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    pf = run_backtest(df, sth_only_entries, trail)
    if pf and pf.trades.count() > 0:
        print(f"{trail*100:>8.0f}% {pf.total_return()*100:>+11.0f}% {pf.sharpe_ratio():>10.2f} {pf.max_drawdown()*100:>9.1f}% {pf.trades.count():>10} {pf.trades.count()/7:>8.1f} {pf.trades.win_rate()*100:>7.0f}%")

In [ ]:
# Trade details for STH-SOPR only (30% trail)
print("\nSTH-SOPR ONLY TRADE LOG (30% trail)")
print("="*100)
print(pf_sth_only.trades.records_readable.to_string())

In [ ]:
# Year by year comparison
print("\nYEAR-BY-YEAR COMPARISON")
print("="*100)
print(f"{'Year':<8} {'STRAT-002':>15} {'STH-SOPR only':>15} {'Buy & Hold':>15} {'Best':>15}")
print("-"*100)

eq_002 = pf_strat002.value()
eq_sth = pf_sth_only.value()

for year in [2019, 2020, 2021, 2022, 2023, 2024, 2025]:
    mask = (df.index >= f'{year}-01-01') & (df.index <= f'{year}-12-31')
    
    y_002 = eq_002[mask]
    y_sth = eq_sth[mask]
    y_price = df[mask]['price']
    
    if len(y_002) > 0 and len(y_sth) > 0:
        r_002 = (y_002.iloc[-1] / y_002.iloc[0] - 1) * 100
        r_sth = (y_sth.iloc[-1] / y_sth.iloc[0] - 1) * 100
        r_bh = (y_price.iloc[-1] / y_price.iloc[0] - 1) * 100
        
        best = 'STRAT-002' if r_002 > r_sth and r_002 > r_bh else ('STH-SOPR' if r_sth > r_bh else 'B&H')
        
        print(f"{year:<8} {r_002:>+14.0f}% {r_sth:>+14.0f}% {r_bh:>+14.0f}% {best:>15}")

In [ ]:
# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

print(f"\n📊 STRAT-002 (SOPR<1 & STH-SOPR<1 & RL Z>0.5):")
print(f"   Return: {pf_strat002.total_return()*100:+,.0f}%")
print(f"   Trades: {pf_strat002.trades.count()} ({pf_strat002.trades.count()/7:.1f}/year)")
print(f"   Sharpe: {pf_strat002.sharpe_ratio():.2f}")

print(f"\n📈 STH-SOPR < 1 only:")
print(f"   Return: {pf_sth_only.total_return()*100:+,.0f}%")
print(f"   Trades: {pf_sth_only.trades.count()} ({pf_sth_only.trades.count()/7:.1f}/year)")
print(f"   Sharpe: {pf_sth_only.sharpe_ratio():.2f}")

print(f"\n💡 Comparison:")
print(f"   More trades: {'✅' if pf_sth_only.trades.count() > pf_strat002.trades.count() else '❌'} ({pf_sth_only.trades.count()} vs {pf_strat002.trades.count()})")
print(f"   Better return: {'✅' if pf_sth_only.total_return() > pf_strat002.total_return() else '❌'}")
print(f"   Better Sharpe: {'✅' if pf_sth_only.sharpe_ratio() > pf_strat002.sharpe_ratio() else '❌'}")

In [ ]:
# Plot both
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=eq_002.index, y=eq_002.values, name='STRAT-002 (triple)', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=eq_sth.index, y=eq_sth.values, name='STH-SOPR only', line=dict(color='green')))

# Add B&H
bh_equity = 100000 * (df['price'] / df['price'].iloc[0])
fig.add_trace(go.Scatter(x=bh_equity.index, y=bh_equity.values, name='Buy & Hold', line=dict(color='gray', dash='dash')))

fig.update_layout(title='STRAT-002 vs STH-SOPR Only', yaxis_title='Equity ($)', yaxis_type='log', height=500)
fig.show()